In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

import os, glob

DATA_DIR = "/content/drive/MyDrive/medicine"

if os.path.isdir(DATA_DIR):
    print("DATA_DIR =", DATA_DIR)
    for p in sorted(glob.glob(os.path.join(DATA_DIR, "*"))):
        print(" -", os.path.basename(p))
else:
    print("Folder not found:", DATA_DIR)
    print("Tip: create it in Drive, or change DATA_DIR to your folder path.")

DATA_DIR = /content/drive/MyDrive/medicine
 - test_set.zip
 - test_set_pixel_size.csv
 - training_set.zip
 - training_set_pixel_size_and_HC.csv


In [ ]:

%pip -q install gdown

import os, gdown

DATA_DIR = "/content/drive/MyDrive/medicine"
os.makedirs(DATA_DIR, exist_ok=True)

FILE_IDS = {
    # "training_set.zip": "PASTE_FILE_ID_HERE",
    # "test_set.zip": "PASTE_FILE_ID_HERE",
    # "training_set_pixel_size_and_HC.csv": "PASTE_FILE_ID_HERE",
    # "test_set_pixel_size.csv": "PASTE_FILE_ID_HERE",
}

for name, fid in FILE_IDS.items():
    out_path = os.path.join(DATA_DIR, name)
    if os.path.exists(out_path):
        print("Exists:", name)
        continue
    print("Downloading:", name)
    gdown.download(id=fid, output=out_path, quiet=False)
print("Done.")

Done.


In [ ]:

%pip install -q torch torchvision pandas pillow scikit-learn matplotlib numpy


In [ ]:

import os, zipfile, random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

SEED = 42
IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 25

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [ ]:

DATA_DIR = "/content/drive/MyDrive/medicine"

TRAIN_ZIP = os.path.join(DATA_DIR, "training_set.zip")
TEST_ZIP  = os.path.join(DATA_DIR, "test_set.zip")

TRAIN_CSV = os.path.join(DATA_DIR, "training_set_pixel_size_and_HC.csv")
TEST_CSV  = os.path.join(DATA_DIR, "test_set_pixel_size.csv")

WORKDIR = "/content/data"
os.makedirs(WORKDIR, exist_ok=True)

print("Data dir:", DATA_DIR)


Data dir: /content/drive/MyDrive/medicine


In [ ]:

def unzip(zip_path, out_dir):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)

unzip(TRAIN_ZIP, WORKDIR)
unzip(TEST_ZIP, WORKDIR)

print("Files in WORKDIR:", os.listdir(WORKDIR))


Files in WORKDIR: ['test_set', 'training_set']


In [ ]:

def _normalize_metadata_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]

    col_map = {}

    # image filename
    if "image_name" not in df.columns:
        for cand in ["filename", "file_name", "image", "img", "name"]:
            if cand in df.columns:
                col_map[cand] = "image_name"
                break

    # pixel size
    if "pixel_size" not in df.columns:
        for cand in ["pixel size(mm)", "pixel_size(mm)", "pixel size", "pixel_size"]:
            if cand in df.columns:
                col_map[cand] = "pixel_size"
                break

    # head circumference (label)
    if "HC" not in df.columns:
        for cand in ["head circumference (mm)", "head_circumference(mm)", "head circumference", "hc", "HC(mm)"]:
            if cand in df.columns:
                col_map[cand] = "HC"
                break

    if col_map:
        df = df.rename(columns=col_map)

    # Final sanity checks
    required = ["image_name", "pixel_size"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"CSV is missing required columns {missing}. Found columns: {list(df.columns)}")

    return df


class HCDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, require_label: bool = True):
        df = pd.read_csv(csv_file)
        df = _normalize_metadata_df(df)

        if require_label and "HC" not in df.columns:
            raise ValueError(f"Training CSV must contain HC/label column. Found columns: {list(df.columns)}")

        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.has_label = "HC" in df.columns

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, str(row["image_name"]))
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        pixel_size = torch.tensor(float(row["pixel_size"]), dtype=torch.float32)

        if self.has_label:
            hc = torch.tensor(float(row["HC"]), dtype=torch.float32)
        else:
            hc = torch.tensor(0.0, dtype=torch.float32)

        return image, pixel_size, hc


In [ ]:

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

train_dataset = HCDataset(TRAIN_CSV, os.path.join(WORKDIR, "training_set"), transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)


In [ ]:

class HCRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, 1)

    def forward(self, x):
        return self.backbone(x).squeeze(1)

model = HCRegressor().to(device)
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 199MB/s]


In [ ]:

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    for imgs, pixel_size, hc in train_loader:
        imgs, hc = imgs.to(device), hc.to(device)

        preds = model(imgs)
        loss = criterion(preds, hc)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} - MAE: {epoch_loss/len(train_loader):.4f}")


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/452_HC.png'

In [ ]:
!ls -la /content
!ls -la /content/data
!find /content/data -maxdepth 3 -type f -name "452_HC.png" | head -n 10
#test file, cant train 452png

total 24
drwxr-xr-x 1 root root 4096 Jan 27 02:57 .
drwxr-xr-x 1 root root 4096 Jan 27 02:56 ..
drwxr-xr-x 4 root root 4096 Jan 16 14:24 .config
drwxr-xr-x 4 root root 4096 Jan 27 02:58 data
drwx------ 5 root root 4096 Jan 27 02:56 drive
drwxr-xr-x 1 root root 4096 Jan 16 14:24 sample_data
total 88
drwxr-xr-x 4 root root  4096 Jan 27 02:58 .
drwxr-xr-x 1 root root  4096 Jan 27 02:57 ..
drwxr-xr-x 2 root root 12288 Jan 27 02:58 test_set
drwxr-xr-x 2 root root 69632 Jan 27 02:58 training_set
/content/data/training_set/452_HC.png
